# Creating your own task

`l2co_tasks` ships with a range of built-in optimization tasks -- BBOB / CEC 2005 benchmark functions, quadratic least-squares, PDE (PINN) problems, and supervised-learning tasks (spiral, MNIST-1D, Gaussian classification) -- each exposed through a `create_<name>_task(...)` factory.

All of them return the same object: a [`Task`](api.md#l2co_tasks.Task). This guide shows how to **build your own** `Task` so it plugs into the rest of the L2CO ecosystem (the [`l2co`](https://github.com/bessagroup/l2co) `RolloutWrapper`, the experiment pipelines, etc.) exactly like a built-in one.

A `Task` bundles four things into one serializable, JAX-friendly object:

1. a **model** -- the PyTree of parameters to optimize,
2. a **loss function** -- what to minimize,
3. (optionally) a **dataset** -- data the loss needs, loaded lazily,
4. **metadata** -- the global minimum, RNG/aux flags, and a free-form `tag`.

## Anatomy of a `Task`

[`Task`](api.md#l2co_tasks.Task) is an `equinox.Module` with the following fields:

| Field | Type | Static? | Meaning |
| --- | --- | --- | --- |
| `model` | `PyTree` | no | PyTree of parameters -- a JAX array or an `eqx.Module`. The **only trainable part**. |
| `loss_fn` | `Callable` | yes | Callable `loss_fn(model, ...)` returning a scalar to minimize. |
| `global_min` | `float \| None` | yes | Minimum of the loss, used for regret/gap reporting (see below). `None` if unknown. |
| `pass_rng` | `bool` | yes | Whether the optimizer must pass a `key=` to `loss_fn` (stochastic losses). |
| `has_aux` | `bool` | yes | Whether `loss_fn` returns `(loss, aux)` instead of just `loss`. |
| `dataset` | `DatasetDict \| None` | yes | `{dataset_path, batch_size, seed}`, or `None` for a data-free task. Arrays are loaded lazily. |
| `tag` | `dict[str, Any]` | yes | Free-form metadata dict; defines task identity (`hash`, `__eq__`). |

### Setting `global_min`

The built-in factories set `global_min` in one of three ways, depending on the problem:

- **analytical** -- a closed-form / retrievable minimum (e.g. the BBOB and CEC 2005 registry optima, the quadratic least-squares residual). A true lower bound on the noiseless loss.
- **theoretical** -- a known limit such as the PDE-residual `0` of the PINN tasks: the loss is a sum of mean-squared-error terms, so `0` is the theoretical minimum (a genuine lower bound, though a finite network need not attain it).
- **empirical** -- when the minimum is unreachable (overlapping classes, or no closed form), the factory runs a short, seeded, multi-restart Adam **benchmark at creation time** and records the best loss found. The public helper [`estimate_global_min`](api.md#l2co_tasks.estimate_global_min) does this for any `Task`; it is deterministic in the task seed, so the value (which feeds the task `hash`) is stable across rebuilds. This is a best-achievable *estimate*, not a guaranteed lower bound.

For your own task, set `global_min` directly if you know it, call `estimate_global_min(task)` to benchmark it, or leave it `None`.

### The `loss_fn` contract

`model` is always the **first positional argument**. How the rest is supplied depends on the flags:

- **Data-free task** (`dataset=None`): called as `loss_fn(model)`.
- **Dataset task**: each entry of the dataset dict is passed as a keyword argument of a mini-batch, i.e. `loss_fn(model, x=..., y=...)` for a dataset with arrays `'x'` and `'y'`.
- **Stochastic loss** (`pass_rng=True`): an RNG key is also passed, as `loss_fn(model, key=..., **batch)`.

`loss_fn` operates on a **single** parameter set (the optimizer vmaps it across the population), so use `jax.vmap(model)(x)` to apply an `eqx.Module` model across a batch of inputs.

!!! warning "Keep `loss_fn` picklable"
    `Task.save` serializes `loss_fn` (and the model skeleton) with `cloudpickle`. Use **module-level functions** or `functools.partial` over picklable values (arrays, floats). Anonymous lambdas closing over non-picklable state will break `save` / `load`.

In [ ]:
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import jax.random as jr

from l2co_tasks import Task

## A minimal task (no dataset)

The simplest task is a closed-form objective over a plain JAX vector. Here is the [Rosenbrock function](https://en.wikipedia.org/wiki/Rosenbrock_function) in 5 dimensions, whose global minimum is `0` at `(1, 1, ..., 1)`.

The model is just the parameter vector, initialised at the origin; the loss is a deterministic function of the model alone, so `dataset`, `pass_rng` and `has_aux` keep their defaults.

In [ ]:
def rosenbrock_loss(model):
    """Rosenbrock objective; minimum 0 at the all-ones vector."""
    x = model
    return jnp.sum(100.0 * (x[1:] - x[:-1] ** 2) ** 2 + (1.0 - x[:-1]) ** 2)


rosenbrock = Task(
    model=jnp.zeros(5),
    loss_fn=rosenbrock_loss,
    global_min=0.0,
    tag={"task_name": "rosenbrock", "dimensionality": 5},
)

rosenbrock

The `Task` exposes a few convenience properties derived from the model and tag:

In [ ]:
print(
    "dimensionality:", rosenbrock.dimensionality
)  # counts float-array leaves
print("name:", rosenbrock.name)  # tag['task_name']
print("hash:", rosenbrock.hash)  # identity from the tag
print()
print("loss at origin:", float(rosenbrock.loss_fn(rosenbrock.model)))
print("loss at optimum:", float(rosenbrock.loss_fn(jnp.ones(5))))

## A task with a dataset

For supervised problems the loss needs data. The model is typically an `eqx.Module`, and the data lives in an `.npz` file that the `Task` references by path and **loads lazily** -- so a task is cheap to construct, copy, and serialize.

Below we fit a small MLP to `y = sin(x)`. Note how the dataset keys (`'x'`, `'y'`) line up with the keyword arguments of the loss function.

In [ ]:
def mse_loss(model, x, y):
    """Mean-squared error of an eqx.Module model over a batch."""
    y_pred = jax.vmap(model)(x)  # apply the model across the batch
    return jnp.mean((y - y_pred) ** 2)


model_key, data_key = jr.split(jr.key(0))

# An eqx.Module model: a small multilayer perceptron.
model = eqx.nn.MLP(
    in_size=1, out_size=1, width_size=32, depth=2, key=model_key
)

# Build and cache the dataset as an .npz file (arrays keyed by name,
# leading axis = number of samples).
xs = jr.uniform(data_key, (256, 1), minval=-3.0, maxval=3.0)
ys = jnp.sin(xs)

dataset_path = Path("sin_dataset.npz")
jnp.savez(dataset_path, x=xs, y=ys)

sin_task = Task(
    model=model,
    loss_fn=mse_loss,
    dataset={
        "dataset_path": str(dataset_path),
        "batch_size": 32,
        "seed": 0,
    },
    tag={"task_name": "sin_regression", "loss_fn": "mse"},
)

sin_task

The dataset is exposed lazily through the `loaded_dataset` property, and the chosen mini-batch size through `batch_size`. The dimensionality now counts every weight in the MLP:

In [ ]:
data = sin_task.loaded_dataset  # reads the .npz on access
print("dataset:", {k: v.shape for k, v in data.items()})
print("batch_size:", sin_task.batch_size)
print("dimensionality:", sin_task.dimensionality)
print("loss (full dataset):", float(sin_task.loss_fn(sin_task.model, **data)))

## Stochastic and multi-output losses

Two flags adapt the loss contract:

- **`pass_rng=True`** -- set this when the loss is stochastic (e.g. injected noise, dropout, a sampled mini-objective). The optimizer then calls `loss_fn(model, key=<PRNGKey>, **batch)`, so your loss must accept a `key` keyword argument.
- **`has_aux=True`** -- set this when `loss_fn` returns `(loss, aux)` rather than a bare scalar; `aux` is any extra information you want to carry alongside the loss.

Everything else -- the model, dataset wiring, and serialization -- is unchanged.

## Saving and loading

`Task.save` writes a single self-contained `.eqx` file: a JSON header (the cloudpickled `loss_fn` and model skeleton, the dataset reference, and the tags) followed by the model's binary weights. `Task.load` reverses it. The dataset `.npz` is **not** copied -- only its path is stored (relative to the `.eqx` file), so a task and its dataset stay portable when moved together.

In [ ]:
saved_path = Task.save(sin_task, "sin_task")
print("saved to:", saved_path)

reloaded = Task.load("sin_task")

# Tasks compare equal by their tag, and the round-trip preserves the loss.
print("reloaded == original:", reloaded == sin_task)
print(
    "loss after reload:",
    float(reloaded.loss_fn(reloaded.model, **reloaded.loaded_dataset)),
)

## Using your task with `l2co`

Because a custom `Task` is indistinguishable from a built-in one, it drops straight into the [`l2co`](https://github.com/bessagroup/l2co) optimization machinery. For example, to run an optimizer on it:

```python
import jax.random as jr
from l2co import OptimizationStep, RolloutWrapper
from l2co.sampling import normal_sampling

rollout = RolloutWrapper.init(
    optimizer=OptimizationStep(
        optimizer='adam', hyperparameters={'learning_rate': 1e-2}
    ),
    task=sin_task,            # <-- your custom task
    sampler=normal_sampling,
    key=jr.key(0),
)
run_state, batch_state, history = rollout.batch_evaluate(
    key=jr.key(1), n_iterations=500, n_realizations=10
)
```

To keep the optimizer inside a box, declare the box on the task rather than passing it to `l2co`: build the task with `constraints=[Box(-5.0, 5.0)]` (`from l2co_tasks import Box`). Every optimizer `l2co` builds for the task clips to it. See *Constraints* in the API reference.

See the [`l2co` Getting Started guide](https://github.com/bessagroup/l2co/blob/main/docs/usage.ipynb) for the full optimization and analysis workflow.

## Summary

To create your own task:

1. **Write a `loss_fn(model, ...)`** that returns a scalar to minimize -- keep it picklable (module-level or `functools.partial`).
2. **Build a `model`** -- a JAX array or `eqx.Module` holding the trainable parameters.
3. For supervised problems, **save the data** to an `.npz` and reference it via `dataset={'dataset_path', 'batch_size', 'seed'}`; the dataset keys become the loss's keyword arguments.
4. **Set the metadata** -- `global_min` (set it directly if known, call `estimate_global_min(task)` to benchmark it, else leave it `None`), `pass_rng` / `has_aux` if the loss needs them, and a descriptive `tag`.
5. **`Task.save` / `Task.load`** to persist it.

The result works anywhere a built-in `l2co_tasks` task does -- including the `l2co` `RolloutWrapper` and the f3dasm experiment pipelines.